# FASE 7 - Multimodal: Audio

Este documento marca o baseline inicial da fase 7 no estado atual do repositorio.


## Objetivo

Construir o nucleo do ScoreIA para entender e manipular audio musical com foco em:

- transcricao;
- separacao de voz e instrumental;
- analise musical;
- criacao de beats por referencia;
- mixagem;
- masterizacao.


## Arquitetura adotada

A fase 7 segue o mesmo principio da fase 6:

- `tools/audio_gen/`: runtime, schemas e persistencia;
- `projects/audio_gen/<project_id>/`: estado duravel do projeto;
- `artifacts/audio_gen/<project_id>/run_<timestamp>/`: artefatos por execucao;
- `models/audio_gen/`: modelos e arquivos locais de apoio.


## Entregas do baseline

Ja implementado neste ciclo:

- `tools/audio_gen/project_paths.py`
  - estrutura canonica de projetos e artefatos de audio;
- `tools/audio_gen/schemas.py`
  - registros de referencias, contexto derivado, beat blueprints, mix sessions, masters e working sets;
- `tools/audio_gen/project_registry.py`
  - criacao e persistencia de projetos por `project_id`;
- `tools/audio_gen/runtime_env.py`
  - diagnostico de `ffmpeg`, `ffprobe`, filtro `whisper`, Demucs e bibliotecas locais;
- `tools/audio_gen/audio_pipeline.py`
  - transcricao via `ffmpeg` + `whisper.cpp`;
  - separacao de stems via Demucs quando disponivel;
  - analise musical via `ffprobe` + NumPy/SciPy;
  - beat blueprint por referencias;
  - renderizacao de beats a partir de samples e padroes;
  - mixagem local via `ffmpeg`;
  - masterizacao local em duas etapas com `loudnorm`.


## Estado atual por feature

- `Transcricao (Whisper)`
  - pronta no codigo;
  - depende de um modelo `whisper.cpp` local em `models/audio_gen/whisper/`.
- `Separacao voz/instrumental`
  - pronta no codigo;
  - depende de Demucs estar ativo na venv ou como CLI.
- `Analise musical`
  - funcional com o ambiente atual que tenha `ffmpeg`, `ffprobe`, `numpy` e `scipy`.
- `Criacao de beats por referencia`
  - baseline entregue como:
    - blueprint derivado das referencias;
    - render de beat por samples/padroes.
- `Mixagem`
  - funcional com `ffmpeg`.
- `Masterizacao`
  - funcional com `ffmpeg`.


## Estrutura prevista de projeto

```text
projects/audio_gen/<project_id>/
  project.json
  references/
    manifest.json
    audio/
    text/
  derived_context/
    manifest.json
    transcripts/
    analyses/
  beat_blueprints/
    manifest.json
    versions/
  mix_sessions/
    manifest.json
  masters/
    manifest.json
  working_sets/
    manifest.json
```


## Proximos passos recomendados

1. Ativar uma venv com `demucs`, `torchaudio` e modelos Whisper locais.
2. Criar uma webapp dedicada de audio, nos moldes de `tools/img_gen/webapp/`.
3. Adicionar presets musicais por genero para beat blueprint, mix e master.
4. Evoluir de beat render baseado em pattern para composicao assistida por referencias mais ricas.


## Campos Default X Campos Inferidos

Nos manifests e artefatos da fase 7, e importante separar:

- `estrutura/schema`: campos canonicos do sistema;
- `defaults`: valores padrao do pipeline quando o usuario nao informa algo;
- `inferidos`: valores calculados a partir das referencias e do audio analisado.

### Campos default do pipeline

Esses campos podem aparecer nos registros porque o codigo define um valor padrao inicial.
Eles nao sao parametros globais imutaveis; sao apenas o valor assumido naquela execucao se nada diferente for informado.

Defaults principais hoje no codigo:

- `sample_rate = 44100`
- `channels = 2`
- `target_peak_db = -1.0`
- `target_lufs = -14.0`
- `true_peak_db = -1.0`
- `loudness_range = 7.0`
- `bars = 4`
- `steps_per_bar = 16`
- `model_name = htdemucs` na separacao
- `target_stem = vocals` na separacao
- `output_format = json` na transcricao
- `language = auto` na transcricao
- `use_gpu = true` na transcricao
- `swing = 0.0` no render de beat
- `role = sample` em camadas sem papel explicito
- `status = draft` em beat blueprints novos
- `origin = user_upload` em referencias novas
- `status = active` em referencias novas

Exemplos pr?ticos:

- `bars` e `steps_per_bar` em `beat_blueprints/manifest.json` podem vir do default se o request nao tiver sobrescrito esses valores.
- `target_peak_db`, `target_lufs` e `true_peak_db` sao defaults tecnicos de mix/master, nao regras permanentes para toda musica.

### Campos inferidos a partir das referencias

Esses campos sao calculados pelo pipeline com base no conteudo analisado e, portanto, mudam conforme o material de entrada.

Inferidos na analise de audio:

- `tempo_estimate_bpm`
- `tempo_estimate_bpm_librosa`
- `spectral_centroid_hz`
- `spectral_rolloff_85_hz`
- `dominant_frequencies_hz`
- `sub_energy_ratio`
- `mid_energy_ratio`
- `high_energy_ratio`
- `mood_tags`
- `dominant_pitch_class`
- `estimated_key`
- `estimated_scale`
- `estimated_key_confidence`
- `harmonic_ratio`
- `percussive_ratio`
- `chroma_profile`
- `melodic_summary`

Inferidos no beat blueprint por referencias:

- `target_bpm` sugerido quando o request nao informa BPM
- `style_tags` enriquecidas a partir de `mood_tags`
- `summary` textual do blueprint
- `arrangement_notes`
- `mix_notes`
- sugestao de camadas e patterns iniciais (`kick`, `snare`, `hat`)
- direcao textual derivada do material de referencia

Inferidos por heuristica simples:

- `role` da referencia a partir do nome do arquivo, por exemplo `kick`, `snare`, `hat`, `bass`, `vocal`, `loop`

### Campos estruturais do schema

Esses nao sao nem defaults criativos nem inferencias musicais. Eles existem para persistencia e rastreabilidade:

- `schema_version`
- `project_id`
- `manifest_type`
- `items`
- `reference_id`
- `context_id`
- `version_id`
- `session_id`
- `master_id`
- `working_set_id`
- `created_at`
- `updated_at`
- `relative_path`
- `artifact_path` dentro de `metadata`
- `source_reference_ids`
- `source_type`
- `source_url`
- `provider`
- `provider_id`

### Leitura correta dos manifests de teste

Quando um arquivo como `projects/audio_gen/phase7-smoke/beat_blueprints/manifest.json` mostra:

- `target_bpm = 92.0`
- `style_tags = [dark, sub-heavy]`
- `arrangement_notes` e `mix_notes`

isso representa o resultado daquela execucao e daquelas referencias, nao um conjunto de parametros fixos para sempre.

Da mesma forma, um arquivo como `derived_context/analyses/tone-reference_analysis.json` guarda descritores do audio analisado naquela execucao especifica.
